**Note: Please "join" the competition first. Then, you can mount the dataset to the GPU. Otherwise, the notebook may encounter an error because it cannot access the dataset until you have joined the competition.**

In [1]:
# The score of the reference solution (by the scientific committee) is 0.83
# Import required packages
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import random
import csv
from tqdm import tqdm
import zipfile
import pandas as pd
import random
random.seed(42)

In [2]:
class CustomDataset(Dataset):
    def __init__(self, data_dir, train_file, mode=None, transform=None):#, model=None, device='cpu'):
        self.transform = transform  # Image transformation pipeline (e.g., resizing, normalization)
        self.images = []  # Stores loaded images (PIL Image objects)
        self.labels = []  # Stores corresponding labels (0 for non-grid, 1 for grid collage)
        self.mode = mode  # Operation mode: "train", "val", or "test"

        # Read and parse the training annotation file
        with open(train_file, "r", encoding="gbk") as file: 
            for i, line in enumerate(file):
                line = line.strip()  # Remove extra spaces/newlines
                if not line:  # Skip empty lines
                    continue
                    
                # Split line into fields (columns)
                fields = [field.strip() for field in line.split(',')]
                
                # Validate field count for training mode
                if len(fields) != 4 and self.mode == "train":
                    raise ValueError("Each line must have 4 fields: id,img_url,label,class")
                
                # Process training data (includes labels)
                if self.mode == "train":
                    img_name, img_url, label_str, class_name = fields
                    label = int(label_str)  # Convert label from string to integer
                    
                    # Load image from local file (supports JPG and PNG)
                    try:
                        img_path = os.path.join(data_dir, f'{img_name}.jpg')
                        image = Image.open(img_path).convert('RGB')  # Ensure 3-channel RGB
                    except:  # If JPG not found, try PNG
                        img_path = os.path.join(data_dir, f'{img_name}.png')
                        image = Image.open(img_path).convert('RGB')

                    self.images.append(image)
                    # self.labels.append(label)
                    # if model!=None:
                        # self.labels.append(1 if model((self.transform(image) if self.transform else image).to(device))>random.random()+0.5 else 0)
                    # else:
                    self.labels.append(0)
                
                # Process validation/test data (no labels)
                elif self.mode == "val" or self.mode == "test":
                    img_name, img_url, class_name = fields  # No label in val/test files
                    
                    # Load image (same as training)
                    try:
                        img_path = os.path.join(data_dir, f'{img_name}.jpg')
                        image = Image.open(img_path).convert('RGB')
                    except:
                        img_path = os.path.join(data_dir, f'{img_name}.png')
                        image = Image.open(img_path).convert('RGB')

                    self.images.append(image)  # Only store images, no labels
                else:
                    raise ValueError("Invalid mode: must be 'train', 'val', or 'test'")
        
        ######################## Data Augmentation ###########################
        '''
        Data augmentation function: generates synthetic grid collages from non-grid images
        This helps balance the dataset (often fewer real grid collages)
        '''
        def augmentation(images, labels, augmentation_sample_num=400, COL=2, ROW=2):
            # Collect all non-grid images (label=0) to use for creating synthetic grids
            neg_images = []
            for (img, label) in zip(images, labels):
                # if label == 0:
                neg_images.append(img)
            
            # Shuffle the non-grid images to ensure randomness
            random.shuffle(neg_images)

            # Lists to store augmented images and their labels
            augmentation_images, augmentation_labels = [], []
            
            # Grid configuration: create 2x2 grids
            # COL = 2  # Number of columns in the grid
            # ROW = 2  # Number of rows in the grid
            UNIT_HEIGHT_SIZE = 128  # Height of each small image in the grid
            UNIT_WIDTH_SIZE = 128   # Width of each small image in the grid
            
            # Generate synthetic grid collages
            # Iterate over non-grid images in steps of 4 (since 2x2 grid needs 4 images)
            for i in range(0, len(neg_images)//(COL*ROW), COL*ROW):
                # Create a blank canvas for the grid collage
                augmentation_image = Image.new(
                    'RGB', 
                    (UNIT_WIDTH_SIZE * COL, UNIT_HEIGHT_SIZE * ROW)  # Total size: 256x256
                )
                
                # Paste each small image into its position in the grid
                for row in range(ROW):
                    for col in range(COL):
                        # Calculate index of the image to paste
                        img_index = i * COL * ROW + COL * row + col
                        # Paste the image at (column*width, row*height)
                        augmentation_image.paste(
                            neg_images[img_index].resize((UNIT_WIDTH_SIZE, UNIT_HEIGHT_SIZE), Image.BILINEAR),
                            (UNIT_WIDTH_SIZE * col, UNIT_HEIGHT_SIZE * row)
                        )
                
                # Add the synthetic grid collage to the dataset
                augmentation_images.append(augmentation_image)
                augmentation_labels.append(1)  # Label synthetic grids as 1

            return augmentation_images, augmentation_labels


        ######################## Apply Augmentation #############################
        # Only augment data during training
        if self.mode == "train":
            for r,c in [(1,2),(2,3),(2,2)]:
                # Generate augmented images and labels
                augmentation_images, augmentation_labels = augmentation(self.images, self.labels, 400, r, c)
                # Add augmented data to the original dataset
                self.images.extend(augmentation_images)
                self.labels.extend(augmentation_labels)

        # self.images = torch.stack(self.images)
        self.labels = torch.tensor(self.labels)
    
    # Return the total number of samples in the dataset
    def __len__(self):
        return len(self.images)

    # Retrieve a single sample by index
    def __getitem__(self, idx):
        image = self.images[idx]  # Get the image at position 'idx'
        
        # Apply transformations if specified (e.g., resizing, normalization)
        if self.transform:
            image = self.transform(image)
            
        # For training, return both image and label; for val/test, return only image
        if self.mode == "train":
            label = self.labels[idx]
            return image, 1 if label>=0.5 else 0
        else:
            return image

    def update(self, model, a=0.05, device:str|torch.device='cpu'):
        if self.mode!="train":
            raise RuntimeError("Expected only training mode to update labels")
        res = []
        for img, _ in self:
            res.append(img)
        # print(self.labels.shape)
        with torch.no_grad():
            out = model(torch.stack(res).to(device)).cpu().squeeze(1)
            # print(out.shape)
            self.labels = self.labels * torch.tensor(1-a) + out * a
        # print(self.labels.shape)

In [3]:
# Create a convolutional neural network model for grid collage classification
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        # Define the sequential model
        self.features = nn.Sequential(
            # Convolutional layer 1: 3 input channels (RGB), 8 output channels, 5x5 kernel
            nn.Conv2d(3, 8, 5),
            nn.ReLU(),
            # Max pooling layer: 2x2 kernel with stride 2 (halves size)
            nn.MaxPool2d(2, 2),
            
            # Convolutional layer 2: 8 input channels, 16 output channels, 5x5 kernel
            nn.BatchNorm2d(8),
            nn.Conv2d(8, 16, 5),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Convolutional layer 3: 16 input channels, 32 output channels, 5x5 kernel
            # nn.BatchNorm2d(16),
            # nn.Conv2d(16, 32, 5),
            # nn.ReLU(),
            # nn.MaxPool2d(2, 2),
            
            # # Convolutional layer 4: 32 input channels, 64 output channels, 5x5 kernel
            # nn.BatchNorm2d(32),
            # nn.Conv2d(32, 64, 5),
            # nn.ReLU(),
            # nn.MaxPool2d(2, 2)
        )
        # Fully connected layers
        self.classifier = nn.Sequential(
            # Fully connected layer 1: 64*12*12 input features → 512 output features
            # nn.BatchNorm(),
            nn.Linear(16 * 61 * 61, 1, bias=False),
            # nn.ReLU(),
            # Fully connected layer 2: 512 input features → 1 output feature (binary classification)
            # nn.BatchNorm(),
            # nn.Linear(512, 1, bias=False),
            # Sigmoid activation: converts output to probability (0-1 range)
            nn.Sigmoid()
        )

    def forward(self, x):
        # Forward pass through first convolutional block: conv1 → ReLU → pool
        if x.dim()==3:
            x=x.unsqueeze(0)
            Squeeze=True
        else:
            Squeeze=False
        x = self.features(x)
        # print(x.shape)
        x = x.view(-1, 16 * 61 * 61)
        x = self.classifier(x)
        if Squeeze==True:
            x=x.squeeze(0)
        return x

In [4]:
# Evaluation function to measure model accuracy on a dataset
def eval_model(model, data_loader, device):
    """
    Evaluate the model's accuracy on a given dataset.
    
    Args:
        model (nn.Module): Trained model to evaluate.
        data_loader (DataLoader): DataLoader containing evaluation samples.
        device (torch.device): Device to run evaluation on (e.g., 'cpu' or 'cuda').
        
    Returns:
        float: Accuracy score (correct predictions / total samples).
    """
    # Set model to evaluation mode (disables dropout/batchnorm)
    model.eval()
    
    # Initialize counters for correct predictions and total samples
    corrects = 0  # Number of correct predictions
    total = 0     # Total number of samples
    
    # Disable gradient computation for efficiency during inference
    with torch.no_grad():
        # Iterate over batches in the data loader
        for inputs, labels in data_loader:
            # Move inputs and labels to the specified device (GPU/CPU)
            inputs = inputs.to(device)
            # Ensure labels are float32 and reshaped to [batch_size, 1]
            labels = labels.to(device).float().view(-1, 1)
            
            # Forward pass: compute model predictions
            outputs = model(inputs)
            
            # Convert model outputs (probabilities) to binary predictions
            # Threshold at 0.5: >=0.5 → 1 (grid), <0.5 → 0 (non-grid)
            preds = outputs >= 0.5
            
            # Count number of correct predictions in this batch
            corrects += torch.sum(preds == labels).item()
            
            # Update total sample count
            total += labels.size(0)
    
    # Calculate overall accuracy
    accuracy = corrects / total
    return accuracy

In [5]:
# Training function to optimize the model parameters
def train_model(model, train_loader, criterion, optimizer, device, num_epochs=10):
    """
    Train the model for a specified number of epochs.
    
    Args:
        model (nn.Module): Model to train.
        train_loader (DataLoader): DataLoader for training data.
        criterion (nn.Module): Loss function (e.g., BCEWithLogitsLoss).
        optimizer (optim.Optimizer): Optimization algorithm (e.g., Adam).
        device (torch.device): Device to run training on (e.g., 'cuda' or 'cpu').
        num_epochs (int): Number of training epochs (default: 10).
    
    Returns:
        None (prints training progress and accuracy).
    """
    max_accuracy = 0  # Track the highest validation accuracy during training
    
    # Training loop for multiple epochs
    for epoch in range(num_epochs):
        # Set model to training mode (enables dropout/batchnorm)
        model.train()
        
        # Initialize running loss to track average loss per epoch
        running_loss = 0.0
        
        # Iterate over batches in the training data loader
        for inputs, labels in tqdm(train_loader):
            # Move inputs and labels to the specified device
            inputs = inputs.to(device)
            labels = labels.to(device).float().view(-1, 1)  # Reshape to [batch_size, 1]
            
            # Zero the parameter gradients (reset optimizer state)
            optimizer.zero_grad()
            
            # Forward pass: compute model predictions
            outputs = model(inputs)
            
            # Compute loss between predictions and ground truth labels
            loss = criterion(outputs, labels)
            
            # Backward pass: compute gradient of loss w.r.t. model parameters
            loss.backward()
            
            # Update model parameters using the computed gradients
            optimizer.step()
            
            # Accumulate total loss for this epoch (weighted by batch size)
            running_loss += loss.item() * inputs.size(0)
            
            # Print progress: current batch and loss
            # print(f"Batch {step}, Loss: {loss.item():.6f}")
        
        # Calculate average loss for this epoch
        epoch_loss = running_loss / len(train_loader.dataset)
        
        # Evaluate model on training data (note: usually done on validation set)
        train_accuracy = eval_model(model, train_loader, device)
        
        # Log training progress
        log_message = (
            f"Epoch {epoch+1}/{num_epochs}, "
            f"Train Loss: {epoch_loss:.4f}, "
            f"Train Accuracy: {train_accuracy:.4f}"
        )
        print(log_message)
        
        # Update maximum accuracy (could be used to save best model)
        if train_accuracy > max_accuracy:
            max_accuracy = train_accuracy
    
    # Uncomment to print the highest accuracy achieved during training
    # print(f"Max Training Accuracy: {max_accuracy:.4f}")

In [6]:
# Define prediction function for generating submission results
def predict(model, loader, device):
    """
    Generate predictions for a dataset using the trained model.
    
    Args:
        model (nn.Module): Trained model for inference.
        loader (DataLoader): DataLoader containing test samples.
        device (torch.device): Device to run inference on (e.g., 'cuda' or 'cpu').
        
    Returns:
        list: Predicted labels (0 or 1) for each sample in the dataset.
    """
    # Set model to evaluation mode (disables dropout/batchnorm)
    model.eval()
    
    # List to store predictions
    preds = []
    
    # Disable gradient computation for faster inference
    with torch.no_grad():
        # Iterate over batches with progress bar (tqdm)
        for batch in tqdm(loader, desc='Predicting'):
            # Move batch to device
            x = batch.to(device)
            
            # Forward pass: compute model outputs
            output = model(x)
            
            # Convert model outputs to binary predictions (0 or 1)
            # Note: This line contains an error! For sigmoid output, use:
            # pred = (output >= 0.5).int()
            # But current code uses argmax, which is incorrect for binary classification with sigmoid.
            pred = torch.argmax(output, dim=1)
            
            # Collect predictions and move to CPU
            preds.extend(pred.cpu().numpy())
    
    return preds

In [7]:
# Save prediction results to CSV format for competition submission
def save_submission_csv(preds, save_name):
    """
    Convert predictions to CSV format required by the competition.
    
    Args:
        preds (list): List of predicted labels (0 or 1).
        save_name (str): Output file name (e.g., "submission.csv").
        
    Format Requirements:
        - Single column with no header
        - No index column
        - Each row contains one prediction (0 or 1)
    """
    # Convert list of predictions to pandas DataFrame
    df = pd.DataFrame(preds)
    
    # Save to CSV with no index and no header (strict competition format)
    df.to_csv(save_name, index=False, header=False)

In [8]:
# Load training dataset
train_dir = '/bohr/train-duyz/v1/train'  # Training image directory
train_file = '/bohr/train-duyz/v1/train.csv'  # Training annotations file

# Data preprocessing pipeline
transform = transforms.Compose([
    transforms.RandomApply([
        transforms.Resize((256, 256)),  # Resize all images to 256x256 (model input size)
        transforms.Pad(padding=10, fill=255)
    ], p=0.5),
    transforms.RandomHorizontalFlip(p=0.2),  # 50% 概率水平翻转
    transforms.RandomVerticalFlip(p=0.2),    # 50% 概率垂直翻转
    transforms.Resize((256, 256)),  # Resize all images to 256x256 (model input size)
    transforms.ToTensor(),  # Convert to tensor (0-1 range)
    transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1,1] for stable training
])

# Initialize training parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use GPU if available
print(f'Using device: {device}')

# Model, loss, and optimizer setup
model = MyModel().to(device)  # Move model to target device
criterion = nn.BCELoss()  # Binary cross-entropy for binary classification
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)  # Adam optimizer with learning rate 1e-4
# Create data loader
# Real images (non-grid) labeled 0; generated grid images labeled 1
train_dataset = CustomDataset(train_dir, train_file, mode="train", transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, num_workers=4, shuffle=True, persistent_workers=True)  # Shuffle for training
epochs = 7  # Number of training epochs

for i in range(4):
    # Train the model
    train_model(model, train_loader, criterion, optimizer, device, num_epochs=epochs)
    train_dataset.update(model, device=device)

# Key task note: 
# Training set classifies beauty products, but test set requires grid/non-grid distinction

Using device: cuda
100%|██████████| 11/11 [00:01<00:00,  7.46it/s]
Epoch 1/20, Train Loss: 0.8536, Train Accuracy: 0.7321
100%|██████████| 11/11 [00:01<00:00, 10.61it/s]
Epoch 2/20, Train Loss: 0.6305, Train Accuracy: 0.7321
100%|██████████| 11/11 [00:01<00:00, 10.45it/s]
Epoch 3/20, Train Loss: 0.4976, Train Accuracy: 0.8272
100%|██████████| 11/11 [00:01<00:00, 10.63it/s]
Epoch 4/20, Train Loss: 0.4201, Train Accuracy: 0.9165
100%|██████████| 11/11 [00:01<00:00, 10.72it/s]
Epoch 5/20, Train Loss: 0.3186, Train Accuracy: 0.9356
100%|██████████| 11/11 [00:01<00:00, 10.69it/s]
Epoch 6/20, Train Loss: 0.2140, Train Accuracy: 0.9209
100%|██████████| 11/11 [00:01<00:00, 10.70it/s]
Epoch 7/20, Train Loss: 0.1423, Train Accuracy: 0.9656
100%|██████████| 11/11 [00:01<00:00, 10.65it/s]
Epoch 8/20, Train Loss: 0.1016, Train Accuracy: 0.9780
100%|██████████| 11/11 [00:01<00:00, 10.66it/s]
Epoch 9/20, Train Loss: 0.0764, Train Accuracy: 0.9839
100%|██████████| 11/11 [00:01<00:00, 10.66it/s]
Epoch 

Error: 

In [ ]:
import matplotlib.pyplot as plt
for i in range(0, 100):
    img, lbl = train_dataset[i]
    if lbl==0:
        image_np = img.permute(1, 2, 0).numpy()
        plt.imshow(image_np)
        plt.title(f"Label: {model(img.to(device)).detach().item():.4f}")  # 在图像上方显示标签
        plt.axis('off')  # 关闭坐标轴
        plt.show()

counter = {}
for i,l in train_dataset:
    counter[l] = counter.get(l,0)+1

print(counter)

In [ ]:
# Save model parameters to avoid retraining during submission
torch.save(model.state_dict(), '/personal/mymodel.pth') 
# /personal is a fixed directory for saving files in the competition environment

# To load the saved model later:
# 1. Instantiate a new model with the same architecture
# model = MyModel()  # Ensure this matches the class used during saving

# 2. Load saved parameters into the model
# model.load_state_dict(torch.load('/personal/NOAI2025_4_model.pth'))

# 3. Move the model to the appropriate device
# model.to(device)

# Notes:
# - Never modify '/personal' in the path
# - The model architecture (MyModel) must be identical when loading
# - Useful for avoiding long retraining during submission

In [ ]:
# Load test sets (errors here are normal as participants can't access val/test addresses directly)
print("Errors below are normal since participants can't read val/test set addresses")

# Get data path from environment variable (provided in competition system)
if os.environ.get('DATA_PATH'):
    DATA_PATH = os.environ.get("DATA_PATH") + "/" 
else:
    DATA_PATH = ""  # Fallback for local testing

# Define paths for validation and test sets
val_dir = DATA_PATH + '/val'       # Validation set directory
val_file = DATA_PATH + '/val.csv'   # Validation set annotations
test_dir = DATA_PATH + '/test'      # Test set directory
test_file = DATA_PATH + '/test.csv' # Test set annotations


# Load validation set (public score - Leaderboard A)
val_dataset = CustomDataset(val_dir, val_file, mode="val", transform=transform)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=True)  # Shuffle not critical for inference

# Load test set (private score - Leaderboard B)
test_dataset = CustomDataset(test_dir, test_file, mode="test", transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)  # Keep order for submission


# Generate predictions using the trained model
val_preds = predict(model, val_loader, device)  # Predictions for Leaderboard A
test_preds = predict(model, test_loader, device)  # Predictions for Leaderboard B


# Generate submission files
# Save validation predictions (Leaderboard A)
save_submission_csv(val_preds, 'submissionA.csv')
# Save test predictions (Leaderboard B)
save_submission_csv(test_preds, 'submissionB.csv')

# Package into zip for submission
with zipfile.ZipFile('submission.zip', 'w') as zipf:
    zipf.write('submissionA.csv')
    zipf.write('submissionB.csv')

# Clean up temporary files
os.remove('submissionA.csv')
os.remove('submissionB.csv')